# 1. Train/Test Split

Before estimating the Bayesian Marketing Mix Model, the dataset is divided into training and testing subsets. Since Marketing Mix Models operate on time-series data, a chronological split is applied instead of random sampling in order to preserve the temporal structure of the observations.

The first 80% of weekly observations are used for model estimation, while the remaining 20% are reserved for out-of-sample evaluation. This setup allows the model to be evaluated on unseen future periods and reflects realistic forecasting conditions in marketing analytics.

The dataset consists of weekly revenue observations together with multiple paid media channels and control variables. Revenue is treated as the dependent variable, while media spend variables are used as the main explanatory variables in the Bayesian MMM specification.


In [ ]:
!pip install pyreadr

In [ ]:
# Core libraries
import os
import requests
import pyreadr
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Create the same folder names used in VSCode
os.makedirs("raw_data", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

# Download Robyn RData file directly from GitHub
url = "https://github.com/facebookexperimental/Robyn/raw/main/R/data/dt_simulated_weekly.RData"

response = requests.get(url)

with open("raw_data/dt_simulated_weekly.RData", "wb") as f:
    f.write(response.content)

# Read RData file
result = pyreadr.read_r("raw_data/dt_simulated_weekly.RData")

# Extract dataset using the same object name as in VSCode
df = result["dt_simulated_weekly"]

# Save a CSV copy using the same file name as in VSCode
df.to_csv("processed_data/dt_simulated_weekly.csv", index=False)

# Load processed dataset
df = pd.read_csv("processed_data/dt_simulated_weekly.csv")

# Convert date column
df["DATE"] = pd.to_datetime(df["DATE"])

# Sort chronologically
df = df.sort_values("DATE").reset_index(drop=True)

# Define variables
target = "revenue"

media_cols = [
    "tv_S",
    "ooh_S",
    "print_S",
    "facebook_S",
    "search_S"
]

control_cols = [
    "competitor_sales_B",
    "newsletter"
]

# Create chronological train-test split
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# Check split sizes
print("Total observations:", len(df))
print("Train observations:", len(train_df))
print("Test observations:", len(test_df))

# Preview
df.head()

In [ ]:
print("Total observations:", len(df))
print("Train observations:", len(train_df))
print("Test observations:", len(test_df))
print(df[["DATE", "revenue", "tv_S", "ooh_S", "print_S", "facebook_S", "search_S", "competitor_sales_B", "newsletter"]].head())

In [ ]:
# Visualize chronological train-test split
plt.figure(figsize=(15, 5))

plt.plot(
    train_df["DATE"],
    train_df[target],
    label="Train"
)

plt.plot(
    test_df["DATE"],
    test_df[target],
    label="Test"
)

plt.title("Chronological Train-Test Split")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()

plt.show()


# Findings

The chronological train-test split preserves the temporal structure of the dataset and avoids information leakage from future observations into the training process. The testing period represents the most recent weeks in the dataset and therefore provides a realistic out-of-sample evaluation setting for the Bayesian Marketing Mix Model.

The visualization shows that the test set maintains similar seasonal patterns and volatility characteristics observed in the training period. This suggests that the holdout data remains representative of the underlying revenue-generating process while still containing unseen future observations for model evaluation.


# 2. Scaling

Bayesian Marketing Mix Models are sensitive to differences in variable magnitudes because Markov Chain Monte Carlo (MCMC) sampling may become unstable when predictors operate on vastly different scales.

To improve numerical stability and sampling efficiency, media spend variables are scaled using MaxAbsScaler, while control variables are standardized using StandardScaler. The target variable is kept on its original scale in this baseline specification.. This transformation preserves the relative structure of the data while normalizing the magnitude of the variables.


In [ ]:
# Scaling
from sklearn.preprocessing import MaxAbsScaler, StandardScaler

# Initialize scalers
media_scaler = MaxAbsScaler()


In [ ]:
# Scaling and model-ready dataset preparation

# Copy train and test sets
train_model_df = train_df.copy()
test_model_df = test_df.copy()

# First diagnostic model: keep newsletter
control_cols = [
    "competitor_sales_B",
    "newsletter"
]

# Add small epsilon to avoid exact zero media values
epsilon = 1e-6
train_model_df[media_cols] = train_model_df[media_cols] + epsilon
test_model_df[media_cols] = test_model_df[media_cols] + epsilon

# Scale media channels
media_scaler = MaxAbsScaler()

train_model_df[media_cols] = media_scaler.fit_transform(
    train_model_df[media_cols]
)

test_model_df[media_cols] = media_scaler.transform(
    test_model_df[media_cols]
)

# Standardize control variables
control_scaler = StandardScaler()

train_model_df[control_cols] = control_scaler.fit_transform(
    train_model_df[control_cols]
)

test_model_df[control_cols] = control_scaler.transform(
    test_model_df[control_cols]
)

# Prepare feature matrices for MMM
X_train = train_model_df[
    ["DATE"] + media_cols + control_cols
].copy()

X_test = test_model_df[
    ["DATE"] + media_cols + control_cols
].copy()

# Prepare target on original scale for now
y_train = train_model_df[target].copy()
y_train.name = "y"

y_test = test_model_df[target].copy()
y_test.name = "y"

# Checks
print("control_cols:", control_cols)
print("X_train columns:", X_train.columns.tolist())
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
X_train.describe()


In [ ]:
# Check scaled values
train_model_df[media_cols + control_cols + [target]].describe()

# Findings

The scaling procedure successfully normalized the media spend variables and the target variable into a comparable numerical range. This transformation improves numerical stability during Bayesian inference and facilitates more efficient MCMC sampling.

The descriptive statistics also reveal that several media channels are sparse, with median values close to zero. This indicates that advertising activity is concentrated in specific periods rather than being continuously active across all weeks, which is consistent with realistic campaign-based marketing behavior.


# 3. PyMC-Marketing Setup

To estimate the Bayesian Marketing Mix Model, the PyMC-Marketing framework is used. PyMC-Marketing is a probabilistic marketing analytics library built on top of PyMC and provides specialized components for Bayesian MMM estimation, including adstock and saturation transformations.

Using PyMC-Marketing enables explicit specification of structural assumptions such as carryover effects and diminishing returns, which are central to the research question of this seminar paper. In particular, the Bayesian MMM baseline incorporates manually specified structural priors that will later be compared against the implicit learned priors of PFN-based models.


In [ ]:
!pip install pymc-marketing


In [ ]:
# Test PyMC-Marketing installation
import pymc_marketing

print(pymc_marketing.__version__)


# Findings

PyMC-Marketing was successfully installed and imported. The installed version is 0.19.4, which provides the required MMM components for Bayesian media mix modeling.

The environment returned a compiler-related warning indicating that PyTensor may not use C-compiled implementations. This does not prevent model estimation, but it may reduce computational speed during sampling.


# 4. Bayesian Model Specification

The Bayesian Marketing Mix Model is specified using explicit structural assumptions regarding advertising carryover effects and diminishing returns.

To capture delayed advertising effects over time, a geometric adstock transformation is applied to the media channels. This transformation models the persistence of advertising impact across multiple weeks and reflects the temporal carryover behavior identified during the exploratory analysis.

To account for nonlinear media effectiveness and diminishing marginal returns, a Logistic saturation transformation is incorporated. This allows the model to represent the realistic phenomenon where incremental revenue gains weaken at higher advertising spend levels.

In addition to paid media channels, control variables are included in the specification to reduce omitted variable bias and improve causal identification.


In [ ]:
# PyMC-Marketing MMM components
from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.mmm import GeometricAdstock
from pymc_marketing.mmm import LogisticSaturation

# Findings

The Bayesian MMM was successfully specified using explicit structural transformations for both carryover effects and diminishing returns.

A geometric adstock transformation was selected to model the delayed persistence of advertising effects across multiple weeks, consistent with the lagged correlation patterns observed during exploratory analysis.

In addition, a Logistic saturation transformation was incorporated to capture nonlinear advertising effectiveness and diminishing marginal returns at higher spend levels.

The resulting specification represents a structurally informed Bayesian benchmark model that can later be compared against PFN-based models relying on implicit learned priors rather than manually defined marketing assumptions.


# 5. Model Fitting

The Bayesian Marketing Mix Model is estimated using Markov Chain Monte Carlo (MCMC) sampling. Bayesian inference allows the model to estimate posterior distributions for all parameters instead of relying on single-point estimates.

This probabilistic framework is particularly valuable in MMM settings because media attribution is inherently uncertain and often affected by temporal dependencies, nonlinearities, and partial identifiability.

The model is fitted on the training dataset only, while the holdout period remains unseen for later out-of-sample evaluation.


In [ ]:
# Import required classes

import arviz as az
from pymc_marketing.prior import Prior

In [ ]:
# Define safer prior configuration
model_config = {
    "intercept": Prior("Normal", mu=0, sigma=2),

    "likelihood": Prior(
        "Normal",
        sigma=Prior("HalfNormal", sigma=2),
        dims="date"
    ),

    "gamma_control": Prior("Normal", mu=0, sigma=2, dims="control"),
    "gamma_fourier": Prior("Laplace", mu=0, b=1, dims="fourier_mode"),

    "adstock_alpha": Prior(
        "Beta",
        alpha=2,
        beta=2,
        dims="channel"
    ),

    "saturation_beta": Prior(
        "HalfNormal",
        sigma=1.5,
        dims="channel"
    ),
}


In [ ]:
# Define Bayesian MMM
mmm = MMM(
    date_column="DATE",
    channel_columns=media_cols,
    control_columns=control_cols,
    target_column="y",
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    model_config=model_config,
)


In [ ]:
print("saturation:", type(mmm.saturation).__name__)
print("adstock:", type(mmm.adstock).__name__)
print("control_cols:", control_cols)
print("X_train columns:", X_train.columns.tolist())

In [ ]:
# Fit Bayesian MMM

idata = mmm.fit(
    X=X_train,
    y=y_train,
    chains=4,
    cores=2,
    draws=1000,
    tune=2000,
    target_accept=0.95,
    random_seed=42,
)


In [ ]:
summary_diag = az.summary(
    idata,
    var_names=[
        "adstock_alpha",
        "saturation_beta",
        "gamma_control"
    ],
    round_to=3
)

summary_diag


In [ ]:
list(idata.posterior.data_vars)


In [ ]:
idata.sample_stats


In [ ]:
print("Divergences:", int(idata.sample_stats["diverging"].sum()))
print("Max tree depth reached:", int(idata.sample_stats["reached_max_treedepth"].sum()))


## Posterior Parameter Summary

The posterior parameter estimates provide insight into the carryover persistence, saturation behavior, and control variable effects captured by the Bayesian MMM.

### Adstock Parameters

The adstock parameters estimate how strongly advertising effects persist across multiple periods.

| Channel | Mean Adstock Alpha |
|---|---|
| TV | 0.241 |
| OOH | 0.572 |
| Print | 0.428 |
| Facebook | 0.540 |
| Search | 0.558 |

OOH, Facebook, and Search exhibit the strongest carryover persistence, suggesting that their advertising effects remain active across several weeks. TV advertising demonstrates comparatively lower persistence, indicating faster decay of advertising influence over time.

---

### Saturation Parameters

The saturation parameters capture diminishing marginal returns from advertising investments.

| Channel | Mean Saturation Beta |
|---|---|
| TV | 0.258 |
| OOH | 0.162 |
| Print | 0.132 |
| Facebook | 0.185 |
| Search | 0.093 |

TV advertising demonstrates the strongest estimated contribution effect, while Search exhibits comparatively weaker marginal contribution effects. These results suggest that media channels differ substantially in their revenue-generating efficiency and nonlinear response behavior.

---

### Control Variable Effects

The control variable estimates indicate that competitor sales remain positively associated with revenue.

| Variable | Mean | HDI 3% | HDI 97% |
|---|---|---|---|
| competitor_sales_B | 0.136 | 0.121 | 0.152 |

The relatively narrow credible interval indicates that the control variable is well identified by the Bayesian MMM and contributes to improved causal interpretability by accounting for external market dynamics.

## 6. Posterior Predictive Evaluation

After confirming stable MCMC convergence, the next step is to evaluate whether the Bayesian MMM can reproduce the observed revenue patterns. Posterior predictive evaluation allows us to compare the model-implied revenue distribution with the actual observed revenue values.

In this first step, posterior predictive samples are generated for the training period. This helps assess whether the fitted model can explain the data it was estimated on before moving to out-of-sample prediction on the test period.

In [ ]:
# Posterior predictive sampling for the training period
# This generates simulated revenue values from the fitted posterior distribution.

posterior_predictive_train = mmm.sample_posterior_predictive(
    X_pred=X_train,
    extend_idata=True,
    random_seed=42,
)

# Inspect updated inference data
idata

### 7. Posterior Predictive Sampling

Posterior predictive samples were successfully generated from the fitted Bayesian MMM. These samples represent simulated revenue values implied by the posterior parameter distributions estimated during MCMC sampling.

The posterior predictive distribution allows evaluation of whether the Bayesian MMM can realistically reproduce the observed revenue dynamics. This step is important because Bayesian models should not only converge statistically, but also generate plausible data patterns consistent with the observed time series.

In [ ]:
# Inspect posterior predictive variables
list(idata.posterior_predictive.data_vars)

In [ ]:
# Extract posterior predictive samples for y
# Dimensions are usually: chain, draw, date

y_pred_train = idata.posterior_predictive["y"]

# Inspect dimensions and shape
print(y_pred_train.dims)
print(y_pred_train.shape)

In [ ]:
# Compute posterior predictive mean and uncertainty intervals
# We average over chains and draws for each date.

y_pred_mean_train = y_pred_train.mean(dim=("chain", "draw"))

y_pred_hdi_train = az.hdi(
    y_pred_train,
    hdi_prob=0.94
)

# Inspect outputs
print("Predicted mean shape:", y_pred_mean_train.shape)
print("HDI shape:", y_pred_hdi_train["y"].shape)

y_pred_hdi_train

### 8. Posterior Predictive Mean and Uncertainty

The posterior predictive distribution was summarized by computing the posterior predictive mean together with 94% Highest Density Intervals (HDIs) for each observation in the training period.

The posterior predictive mean represents the expected revenue implied by the Bayesian MMM, while the HDI intervals capture posterior uncertainty around the model predictions.

These intervals are useful for evaluating whether the observed revenue series falls within the plausible range generated by the Bayesian posterior distribution.

In [ ]:
# Convert posterior predictive outputs to NumPy arrays

y_pred_mean_train_np = y_pred_mean_train.values

y_pred_lower_train = y_pred_hdi_train["y"].sel(hdi="lower").values
y_pred_upper_train = y_pred_hdi_train["y"].sel(hdi="higher").values

# Actual observed revenue
y_actual_train = y_train.values

# Check shapes
print("Actual:", y_actual_train.shape)
print("Predicted mean:", y_pred_mean_train_np.shape)
print("Lower HDI:", y_pred_lower_train.shape)
print("Upper HDI:", y_pred_upper_train.shape)

### 9. Actual vs Predicted Revenue

The posterior predictive mean is compared against the observed revenue series in order to evaluate how well the Bayesian MMM reproduces the underlying revenue dynamics.

In addition to the posterior predictive mean, 94% Highest Density Intervals (HDIs) are visualized to illustrate predictive uncertainty. A well-specified Bayesian MMM should capture the general trend and seasonal movement of the observed revenue while maintaining reasonable uncertainty bounds.

In [ ]:
print(y_pred_mean_train_np[:10])
print(y_actual_train[:10])

In [ ]:
# Rescale posterior predictive mean and HDI back to original revenue scale

revenue_scale = y_train.max()

y_pred_mean_train_rescaled = y_pred_mean_train_np * revenue_scale

y_pred_lower_train_rescaled = y_pred_lower_train * revenue_scale
y_pred_upper_train_rescaled = y_pred_upper_train * revenue_scale

print("Revenue scale:", revenue_scale)

print(y_pred_mean_train_rescaled[:5])

In [ ]:
# Plot actual vs posterior predictive mean on original revenue scale

plt.figure(figsize=(15, 6))

# Actual revenue
plt.plot(
    train_model_df["DATE"],
    y_actual_train,
    label="Actual Revenue"
)

# Rescaled posterior predictive mean
plt.plot(
    train_model_df["DATE"],
    y_pred_mean_train_rescaled,
    label="Posterior Predictive Mean"
)

# Rescaled 94% HDI interval
plt.fill_between(
    train_model_df["DATE"],
    y_pred_lower_train_rescaled,
    y_pred_upper_train_rescaled,
    alpha=0.3,
    label="94% HDI"
)

plt.title("Posterior Predictive Check - Training Period")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.legend()

plt.show()

### Posterior Predictive Evaluation

The posterior predictive evaluation indicates that the Bayesian MMM is capable of reproducing the main revenue dynamics observed in the training period.

The posterior predictive mean closely follows the observed revenue series across time, successfully capturing the general trend, cyclical movement, and seasonal fluctuations in the data. In addition, the 94% Highest Density Intervals (HDIs) cover the majority of the observed revenue observations, suggesting that the posterior uncertainty estimates remain statistically reasonable.

Although several sharp revenue spikes are not perfectly reproduced, the model still captures the overall temporal structure of the revenue-generating process. This behavior is expected in Bayesian MMM settings because the model prioritizes stable structural attribution and uncertainty estimation rather than exact point prediction of extreme observations.

Overall, the posterior predictive checks suggest that the Bayesian MMM provides a statistically plausible representation of the observed revenue series and is suitable as a benchmark model for subsequent PFN comparisons.

## 10. Out-of-Sample Prediction

After evaluating the posterior predictive performance on the training period, the Bayesian MMM is next evaluated on unseen future observations from the test period.

This step is important because Marketing Mix Models should not only fit historical data, but also generalize to future periods while maintaining stable predictive behavior. The chronological holdout split therefore allows assessment of the model’s out-of-sample forecasting capability under realistic temporal conditions.

In [ ]:
# Generate posterior predictive samples for the test period

posterior_predictive_test = mmm.sample_posterior_predictive(
    X_pred=X_test,
    extend_idata=False,
    random_seed=42,
)

# Inspect posterior predictive test dimensions
posterior_predictive_test

### Test Set Posterior Predictions

Posterior predictive samples were generated for the unseen test period in order to evaluate the out-of-sample forecasting capability of the Bayesian MMM.

The resulting posterior predictive distribution allows comparison between observed future revenue values and the model-implied predictions. In addition to the posterior predictive mean, uncertainty intervals are computed to assess the stability and reliability of the model forecasts.

In [ ]:
# Compute posterior predictive mean and HDI for the test period

y_pred_test = posterior_predictive_test["y"]

# Posterior predictive mean over all posterior samples
y_pred_mean_test = y_pred_test.mean(dim="sample")

# 94% HDI over the sample dimension
y_pred_hdi_test = az.hdi(
    y_pred_test,
    hdi_prob=0.94,
    input_core_dims=[["sample"]]
)

# Inspect dimensions
print("Predicted mean shape:", y_pred_mean_test.shape)
print("HDI shape:", y_pred_hdi_test["y"].shape)

y_pred_hdi_test

In [ ]:
# Convert test posterior predictions back to original revenue scale

y_pred_mean_test_np = y_pred_mean_test.values

y_pred_lower_test = y_pred_hdi_test["y"].sel(hdi="lower").values
y_pred_upper_test = y_pred_hdi_test["y"].sel(hdi="higher").values

# Rescale predictions to original revenue units
y_pred_mean_test_rescaled = y_pred_mean_test_np * revenue_scale
y_pred_lower_test_rescaled = y_pred_lower_test * revenue_scale
y_pred_upper_test_rescaled = y_pred_upper_test * revenue_scale

# Actual test revenue
y_actual_test = y_test.values

# Check shapes and first predictions
print("Actual test:", y_actual_test.shape)
print("Predicted mean:", y_pred_mean_test_rescaled.shape)
print("First 5 predicted:", y_pred_mean_test_rescaled[:5])
print("First 5 actual:", y_actual_test[:5])

In [ ]:
# Plot actual vs posterior predictive mean for the test period

plt.figure(figsize=(15, 6))

plt.plot(
    test_model_df["DATE"],
    y_actual_test,
    label="Actual Revenue"
)

plt.plot(
    test_model_df["DATE"],
    y_pred_mean_test_rescaled,
    label="Posterior Predictive Mean"
)

plt.fill_between(
    test_model_df["DATE"],
    y_pred_lower_test_rescaled,
    y_pred_upper_test_rescaled,
    alpha=0.3,
    label="94% HDI"
)

plt.title("Out-of-Sample Posterior Prediction - Test Period")
plt.xlabel("Date")
plt.ylabel("Revenue")

plt.legend()

plt.show()

### Out-of-Sample Forecast Evaluation

The out-of-sample posterior predictive evaluation indicates that the Bayesian MMM generalizes reasonably well to unseen future observations.

The posterior predictive mean successfully captures the overall temporal dynamics of the test period, including the downward movement during the early weeks and the subsequent recovery trend observed later in the holdout period. In addition, the 94% Highest Density Intervals (HDIs) cover the majority of the observed revenue values, suggesting that the model maintains statistically plausible uncertainty estimates outside the training sample.

Although several short-term spikes are smoothed by the model, the Bayesian MMM still reproduces the broader seasonal and structural revenue patterns with relatively high stability. This behavior is expected because MMMs are primarily designed for robust attribution and structural effect estimation rather than exact prediction of highly volatile short-term fluctuations.

Overall, the out-of-sample results suggest that the Bayesian MMM demonstrates stable forecasting behavior and provides a credible benchmark for subsequent PFN-based comparisons.

## 11. Forecast Accuracy Metrics

In addition to visual posterior predictive evaluation, quantitative forecast accuracy metrics are computed for the test period.

These metrics provide a numerical assessment of the Bayesian MMM’s out-of-sample predictive performance and allow evaluation of prediction error magnitude under realistic future forecasting conditions.

Three commonly used forecasting metrics are reported:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Percentage Error (MAPE)

Together, these metrics provide complementary perspectives on prediction stability and forecast reliability.

In [ ]:
# Forecast accuracy metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

# MAE
mae = mean_absolute_error(
    y_actual_test,
    y_pred_mean_test_rescaled
)

# RMSE
rmse = np.sqrt(
    mean_squared_error(
        y_actual_test,
        y_pred_mean_test_rescaled
    )
)

# MAPE
mape = mean_absolute_percentage_error(
    y_actual_test,
    y_pred_mean_test_rescaled
) * 100

# Print results
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")

### Forecast Accuracy Results

The quantitative forecast evaluation indicates that the Bayesian MMM achieves strong out-of-sample predictive performance on the holdout test period.

| Metric | Value |
|---|---|
| MAE | 115,134.78 |
| RMSE | 218,368.30 |
| MAPE | 8.53% |

The Mean Absolute Percentage Error (MAPE) of 8.53% suggests that the Bayesian MMM maintains relatively low forecasting error despite the presence of seasonal variation and short-term volatility in the revenue series.

Similarly, the MAE and RMSE values indicate that the model predictions remain reasonably close to the observed revenue values throughout the unseen test period. While several sharp spikes are partially smoothed, the model successfully captures the broader structural and seasonal revenue dynamics.

Overall, the forecast accuracy metrics support the conclusion that the Bayesian MMM provides stable and reliable out-of-sample predictions, reinforcing its suitability as the benchmark model for later PFN comparisons.

# 12. Channel Contribution Analysis

One of the primary objectives of Marketing Mix Modeling is not only forecasting revenue, but also decomposing revenue into channel-specific contributions.

The Bayesian MMM estimates posterior distributions for the contribution of each media channel over time. These estimated contributions reflect the model-implied incremental revenue associated with each advertising channel after accounting for adstock carryover effects, saturation behavior, seasonality, and control variables.

This section therefore evaluates how the Bayesian MMM allocates revenue attribution across the different marketing channels.

In [ ]:
# Inspect available posterior variables related to contribution analysis

list(idata.posterior.data_vars)

In [ ]:
# Inspect channel contribution dimensions and shape

channel_contribution = idata.posterior["channel_contribution"]

print(channel_contribution.dims)
print(channel_contribution.shape)
print(channel_contribution.coords)

In [ ]:
# Compute posterior mean channel contributions

channel_contribution_mean = channel_contribution.mean(
    dim=("chain", "draw")
)

# Convert to dataframe
channel_contribution_df = (
    channel_contribution_mean
    .to_pandas()
)

# Inspect contribution dataframe
channel_contribution_df.head()

In [ ]:
# Compute total contribution per channel over the full training period

total_channel_contribution = (
    channel_contribution_df.sum(axis=0)
    .sort_values(ascending=False)
)

# Display total contributions
total_channel_contribution

In [ ]:
# Plot total channel contributions

plt.figure(figsize=(10, 6))

total_channel_contribution.plot(
    kind="bar"
)

plt.title("Total Channel Contribution")
plt.xlabel("Marketing Channel")
plt.ylabel("Total Contribution")

plt.show()

### Channel Contribution Results

The Bayesian MMM estimates indicate substantial differences in revenue contribution across the marketing channels.

| Channel | Total Contribution |
|---|---|
| tv_S | 4.57 |
| facebook_S | 3.28 |
| search_S | 2.46 |
| ooh_S | 1.83 |
| print_S | 1.67 |

TV advertising exhibits the largest estimated contribution to revenue, suggesting that the channel plays the most influential role in the modeled revenue-generating process. Facebook and Search also demonstrate substantial contributions, indicating that digital advertising channels account for a meaningful share of incremental revenue generation.

In contrast, OOH and Print contribute comparatively less to the overall revenue decomposition. This result is broadly consistent with the previously estimated saturation and adstock parameters, where TV and Facebook displayed stronger carryover persistence and contribution magnitudes.

Overall, the channel contribution analysis suggests that the Bayesian MMM produces interpretable and economically plausible attribution patterns across the different marketing channels.

# 13. ROAS Estimation

Return on Advertising Spend (ROAS) is one of the central outputs of Marketing Mix Modeling because it provides an interpretable measure of marketing efficiency across channels.

The Bayesian MMM allows estimation of channel-specific ROAS by comparing the model-implied revenue contribution of each channel against the corresponding advertising spend.

This analysis therefore evaluates which marketing channels generate the highest incremental return relative to their investment level.

In [ ]:
# Compute total media spend per channel

total_spend = train_model_df[media_cols].sum()

print(total_spend)

In [ ]:
# Compute ROAS per channel

roas = total_channel_contribution / total_spend

# Sort descending
roas = roas.sort_values(ascending=False)

# Display ROAS
roas

In [ ]:
# Plot channel ROAS estimates

plt.figure(figsize=(10, 6))

roas.plot(
    kind="bar"
)

plt.title("Estimated ROAS by Channel")
plt.xlabel("Marketing Channel")
plt.ylabel("ROAS")

plt.show()

### ROAS Results

The estimated ROAS values reveal notable differences in marketing efficiency across the advertising channels.

| Channel | Estimated ROAS |
|---|---|
| tv_S | 0.261 |
| facebook_S | 0.126 |
| ooh_S | 0.105 |
| print_S | 0.080 |
| search_S | 0.048 |

TV advertising achieves the highest estimated ROAS, indicating that the channel generates the strongest incremental revenue contribution relative to advertising spend. Facebook also demonstrates comparatively strong efficiency, suggesting that digital advertising contributes meaningfully to revenue generation while maintaining relatively moderate investment levels.

Although Search produces substantial total contribution, its ROAS remains comparatively low due to the considerably larger overall spend allocated to the channel. Similarly, OOH and Print exhibit weaker efficiency relative to TV and Facebook.

Overall, the ROAS estimates suggest that the Bayesian MMM produces economically interpretable efficiency rankings across channels and provides a useful benchmark for evaluating whether PFN-based approaches can recover comparable attribution and ROAS patterns without explicitly specified structural priors.

In [ ]:
# Plot channel contribution decomposition over time

plt.figure(figsize=(16, 7))

plt.stackplot(
    channel_contribution_df.index,
    channel_contribution_df.T,
    labels=channel_contribution_df.columns
)

plt.title("Channel Contribution Decomposition Over Time")
plt.xlabel("Date")
plt.ylabel("Contribution")

plt.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1)
)

plt.show()

### Temporal Contribution Decomposition

The channel contribution decomposition reveals substantial time variation in the estimated media effects across the training period.

TV advertising consistently accounts for a large share of total contribution and frequently dominates the overall revenue attribution during major campaign periods. Facebook also exhibits several pronounced contribution spikes, suggesting periods of intensified digital advertising effectiveness.

In contrast, Search contributions appear comparatively more stable across time, while OOH and Print contribute smaller but persistent effects throughout the observation window.

Importantly, the decomposition illustrates that media effects are not constant over time. Instead, the Bayesian MMM captures dynamic temporal variation in channel influence, which reflects the combined impact of changing advertising intensity, adstock carryover behavior, and saturation effects.

Overall, the temporal contribution analysis further supports the interpretability of the Bayesian MMM and demonstrates its ability to produce economically meaningful time-varying attribution estimates.

# Bayesian MMM Summary

The Bayesian Marketing Mix Model (MMM) was implemented using the PyMC-Marketing framework with explicit structural assumptions regarding carryover effects and diminishing returns. A chronological train-test split was applied in order to preserve the temporal structure of the weekly marketing data and avoid information leakage from future observations.

To improve numerical stability during Bayesian inference, media variables were normalized using MaxAbs scaling, while the continuous control variable was standardized. The Bayesian MMM was then specified using a Geometric Adstock transformation to model delayed advertising effects and a Logistic Saturation transformation to capture diminishing marginal returns at higher spending levels.

The model was estimated using Markov Chain Monte Carlo (MCMC) sampling with four chains and 1,000 posterior draws per chain. The convergence diagnostics indicate stable posterior estimation behavior:

| Diagnostic | Result |
|---|---|
| Divergences | 0 |
| Maximum Tree Depth Reached | 0 |
| R-hat Range | 1.000 – 1.003 |
| Effective Sample Sizes | High across parameters |

These results suggest that the posterior distributions were sampled efficiently and that the Bayesian MMM achieved reliable convergence.

The estimated adstock parameters indicate meaningful carryover persistence across channels, particularly for OOH, Facebook, and Search advertising. In addition, the saturation parameters confirm diminishing returns behavior, with TV advertising exhibiting the strongest estimated contribution effects among the media channels.

Posterior predictive evaluation further demonstrated that the Bayesian MMM reproduces the main temporal revenue dynamics observed in the training period. The posterior predictive mean closely follows the observed revenue series, while the 94% Highest Density Intervals (HDIs) capture the majority of observed observations.

The out-of-sample evaluation on the unseen test period also produced strong forecasting performance:

| Metric | Value |
|---|---|
| MAE | 115,134.78 |
| RMSE | 218,368.30 |
| MAPE | 8.53% |

The relatively low forecasting error suggests that the Bayesian MMM generalizes well to future periods and does not exhibit severe overfitting behavior.

The channel contribution analysis revealed clear differences in estimated marketing effectiveness across channels. TV advertising generated the largest overall contribution, followed by Facebook and Search advertising. Temporal contribution decomposition additionally showed that media effects vary substantially across time, indicating that channel influence is dynamic rather than constant.

The estimated ROAS values further demonstrated meaningful differences in marketing efficiency:

| Channel | Estimated ROAS |
|---|---|
| tv_S | 0.261 |
| facebook_S | 0.126 |
| ooh_S | 0.105 |
| print_S | 0.080 |
| search_S | 0.048 |

TV advertising achieved the highest estimated ROAS, suggesting that it generates the strongest incremental revenue relative to spend. Facebook also demonstrated comparatively efficient performance, while Search generated lower efficiency despite relatively high overall contribution due to larger advertising expenditures.

Overall, the Bayesian MMM produced stable convergence behavior, strong out-of-sample forecasting performance, interpretable attribution patterns, and economically plausible ROAS estimates. The model therefore provides a structurally informed benchmark suitable for subsequent comparison against PFN-based approaches that rely on implicit learned priors rather than explicitly specified marketing assumptions.